# 립리딩 전처리 · 학습 파이프라인 (Colab)

원본 영상을 Google Drive에 두고, 코랩 GPU로 전처리와 학습을 수행한다.

**실행 전 준비**
1. 런타임 → 런타임 유형 변경 → 하드웨어 가속기 **GPU** 선택
2. Drive에 영상 폴더 생성 후 녹화본 업로드
3. 파일명 규칙: `{화자}_{문구}_{번호}.mp4` — 예) `s01_물주세요_01.mp4`

화자가 **2명 이상**이어야 학습이 진행된다. 화자 단위로 학습·검증을 나누기 때문이다.

## 1. 환경 확인

In [54]:
import torch

print(f"torch {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("경고: 런타임 유형을 GPU로 변경하세요.")

torch 2.11.0+cu128
CUDA 사용 가능: True
GPU: NVIDIA A100-SXM4-40GB


## 2. Drive 마운트

In [55]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. 경로 설정

`DRIVE_ROOT` 아래 `raw/`에 영상을 두면, 전처리 결과가 `processed/`에 저장된다.
Drive에 저장하므로 런타임이 끊겨도 전처리를 다시 하지 않아도 된다.

In [56]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/hanium-lipreading")
DRIVE_RAW = DRIVE_ROOT / "raw"
DRIVE_PROCESSED = DRIVE_ROOT / "processed"
DRIVE_CHECKPOINTS = DRIVE_ROOT / "checkpoints"

for folder in (DRIVE_RAW, DRIVE_PROCESSED, DRIVE_CHECKPOINTS):
    folder.mkdir(parents=True, exist_ok=True)

videos = sorted(
    p.name for p in DRIVE_RAW.glob("*") if p.suffix.lower() in (".mp4", ".avi", ".mov")
)
print(f"영상 {len(videos)}개")
for name in videos[:10]:
    print(f"  {name}")

영상 1234개
  s01_가래가있어요_01.mp4
  s01_가래가있어요_02.mp4
  s01_가래가있어요_03.mp4
  s01_가래가있어요_04.mp4
  s01_가래가있어요_05.mp4
  s01_가래가있어요_06.mp4
  s01_가래가있어요_07.mp4
  s01_가래가있어요_08.mp4
  s01_가래가있어요_09.mp4
  s01_가래가있어요_10.mp4


## 4. 저장소 clone

이미 받아둔 저장소가 있으면 `git pull`로 최신 코드만 가져온다.
clone이 중간에 실패해 빈 폴더가 남은 경우에는 지우고 다시 받는다.

비공개 저장소면 `https://<TOKEN>@github.com/...` 형태로 토큰을 넣는다.
토큰은 노트북에 저장하지 말고 매번 입력한다.

**코드를 갱신한 뒤에는 런타임을 다시 시작하거나 모듈을 reload해야 반영된다.**
파이썬은 한 번 import한 모듈을 다시 읽지 않는다.

```python
import importlib
import src.ml.training.dataset, src.ml.training.train
importlib.reload(src.ml.training.dataset)
importlib.reload(src.ml.training.train)
from src.ml.training.train import train
```

In [57]:
import os

REPO_URL = "https://github.com/HumanRhoid/hanium-lipreading.git"
BRANCH = "develop"
REPO_DIR = Path("/content/hanium-lipreading")

# clone이 중간에 실패하면 빈 폴더만 남아 다음 실행에서 git 명령이 어긋난다.
os.chdir("/content")
if (REPO_DIR / ".git").exists():
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull
else:
    !rm -rf {REPO_DIR}
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"작업 경로: {Path.cwd()}")

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 5 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 52.81 KiB | 1.20 MiB/s, done.
From https://github.com/HumanRhoid/hanium-lipreading
   7e96c73..5fa34ab  develop                -> origin/develop
 * [new branch]      feature/colab_pipeline -> origin/feature/colab_pipeline
Already on 'develop'
Your branch is behind 'origin/develop' by 2 commits, and can be fast-forwarded.
  (use "git pull" to update your local branch)
Updating 7e96c73..5fa34ab
Fast-forward
 notebooks/colab_pipeline.ipynb | 8612 ++++++++++++++++++++++++++--------------
 1 file changed, 5690 insertions(+), 2922 deletions(-)
작업 경로: /content/hanium-lipreading


## 5. 의존성 설치

`uv sync`는 쓰지 않는다. `pyproject.toml`이 torch를 **CPU 전용 인덱스**로 고정하고 있어
코랩의 GPU torch가 CPU 버전으로 교체되기 때문이다. 필요한 것만 pip로 설치한다.

In [58]:
!pip install --quiet mediapipe opencv-python wandb

import torch

print(f"설치 후 CUDA 사용 가능: {torch.cuda.is_available()}")

Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 259, in _feed
    reader_close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 178, in close
    self._close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor


설치 후 CUDA 사용 가능: True


## 6. 얼굴 랜드마크 모델 내려받기

`face_landmarker.task`는 `.gitignore`에 제외돼 있어 저장소에 없다.

In [59]:
LANDMARKER_URL = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
landmarker_path = REPO_DIR / "models" / "face_landmarker.task"
landmarker_path.parent.mkdir(parents=True, exist_ok=True)

if not landmarker_path.exists():
    !wget -q -O {landmarker_path} {LANDMARKER_URL}

print(f"{landmarker_path.name}: {landmarker_path.stat().st_size / 1e6:.1f} MB")

face_landmarker.task: 3.8 MB


## 7. 전처리 — 영상을 .npy로 변환

Drive를 입출력으로 직접 지정한다. 이미 변환된 파일은 건너뛴다.

In [60]:
import sys

sys.path.insert(0, str(REPO_DIR))

from src.ml.preprocess.vid2npy import run_batch

run_batch(raw_dir=DRIVE_RAW, processed_dir=DRIVE_PROCESSED)

이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_01.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_02.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_03.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_04.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_05.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_06.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_07.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_08.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_09.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_10.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_11.npy
이미 존재함, 건너뜀: /content

In [61]:
FRAMES = 45
PROCESSED_F45 = DRIVE_ROOT / "processed_f45"

from src.ml.preprocess import vid2npy
from src.ml.preprocess.lip_crop import crop_lip_frames
from src.ml.preprocess.normalize import normalize_frames

# normalize_frames의 기본값은 def 시점에 고정되므로 함수를 갈아끼운다
def process_video_f45(video_path, landmarker):
    lips, opennesses = crop_lip_frames(video_path, landmarker)
    if not lips:
        return None
    return normalize_frames(lips, opennesses, fixed_frame_count=FRAMES)

vid2npy.process_video = process_video_f45
vid2npy.run_batch(raw_dir=DRIVE_RAW, processed_dir=PROCESSED_F45)

이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_01.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_02.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_03.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_04.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_05.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_06.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_07.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_08.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_09.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s01_가래가있어요_10.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f45/s0

In [62]:
FRAMES = 60
PROCESSED_F60 = DRIVE_ROOT / "processed_f60"

from src.ml.preprocess import vid2npy
from src.ml.preprocess.lip_crop import crop_lip_frames
from src.ml.preprocess.normalize import normalize_frames

def process_video_f60(video_path, landmarker):
    lips, opennesses = crop_lip_frames(video_path, landmarker)
    if not lips:
        return None
    return normalize_frames(lips, opennesses, fixed_frame_count=FRAMES)

vid2npy.process_video = process_video_f60
vid2npy.run_batch(raw_dir=DRIVE_RAW, processed_dir=PROCESSED_F60)

이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_01.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_02.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_03.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_04.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_05.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_06.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_07.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_08.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_09.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_10.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s0

## 8. 매니페스트 생성

`.npy` 파일명을 파싱해 라벨과 화자를 뽑아낸다.

In [63]:
from scripts.build_manifest import build

manifest_path = DRIVE_ROOT / "manifest.csv"
build(processed_dir=DRIVE_PROCESSED, manifest_path=manifest_path)

매니페스트 생성: /content/drive/MyDrive/hanium-lipreading/manifest.csv
  클립 1234개 · 문구 15개 · 화자 8명
  라벨 매핑: 0=가래가있어요, 1=간호사불러주세요, 2=더워요, 3=도와주세요, 4=물주세요, 5=배고파요, 6=보호자불러주세요, 7=숨쉬기힘들어요, 8=아파요, 9=어지러워요, 10=자세바꿔주세요, 11=진통제주세요, 12=추워요, 13=토할거같아요, 14=화장실가고싶어요


In [64]:
import csv
from collections import Counter

with open(manifest_path, encoding="utf-8") as manifest_file:
    rows = list(csv.DictReader(manifest_file))

print(f"클립 {len(rows)}개")
print(f"화자별: {dict(Counter(row['speaker_id'] for row in rows))}")
print(f"문구별: {dict(Counter(row['label_text'] for row in rows))}")

클립 1234개
화자별: {'s01': 157, 's03': 148, 's04': 150, 's05': 150, 's06': 157, 's07': 171, 's08': 151, 's09': 150}
문구별: {'가래가있어요': 80, '간호사불러주세요': 79, '더워요': 86, '도와주세요': 82, '물주세요': 88, '배고파요': 85, '보호자불러주세요': 82, '숨쉬기힘들어요': 82, '아파요': 79, '어지러워요': 80, '자세바꿔주세요': 83, '진통제주세요': 86, '추워요': 82, '토할거같아요': 80, '화장실가고싶어요': 80}


### (45 프레임)


In [65]:
from scripts.build_manifest import build

manifest_f45 = DRIVE_ROOT / "manifest_f45.csv"
build(processed_dir=PROCESSED_F45, manifest_path=manifest_f45)

매니페스트 생성: /content/drive/MyDrive/hanium-lipreading/manifest_f45.csv
  클립 1234개 · 문구 15개 · 화자 8명
  라벨 매핑: 0=가래가있어요, 1=간호사불러주세요, 2=더워요, 3=도와주세요, 4=물주세요, 5=배고파요, 6=보호자불러주세요, 7=숨쉬기힘들어요, 8=아파요, 9=어지러워요, 10=자세바꿔주세요, 11=진통제주세요, 12=추워요, 13=토할거같아요, 14=화장실가고싶어요


### (60프레임 매니페스트 및 로컬 복사 및 검증포함)


In [66]:
from scripts.build_manifest import build
from pathlib import Path
import shutil, time, numpy as np

manifest_f60 = DRIVE_ROOT / "manifest_f60.csv"
build(processed_dir=PROCESSED_F60, manifest_path=manifest_f60)

LOCAL_F60 = Path("/content/data60")
target = LOCAL_F60 / "processed"
expected = len(list(PROCESSED_F60.glob("*.npy")))

if target.exists() and len(list(target.glob("*.npy"))) != expected:
    shutil.rmtree(target)
if not target.exists():
    t = time.time()
    shutil.copytree(PROCESSED_F60, target)
    print(f"복사 {time.time() - t:.0f}초")

# 아까 겪은 0바이트·누락 확인
bad = [p.name for p in target.glob("*.npy")
       if p.stat().st_size == 0 or (np.load(p, mmap_mode="r") is None)]
n = len(list(target.glob("*.npy")))
print(f"로컬 {n}/{expected}개 · 손상 {len(bad)}개")
assert n == expected and not bad, "복사가 불완전합니다"

TRAIN_ROOT_F60 = LOCAL_F60

매니페스트 생성: /content/drive/MyDrive/hanium-lipreading/manifest_f60.csv
  클립 1234개 · 문구 15개 · 화자 8명
  라벨 매핑: 0=가래가있어요, 1=간호사불러주세요, 2=더워요, 3=도와주세요, 4=물주세요, 5=배고파요, 6=보호자불러주세요, 7=숨쉬기힘들어요, 8=아파요, 9=어지러워요, 10=자세바꿔주세요, 11=진통제주세요, 12=추워요, 13=토할거같아요, 14=화장실가고싶어요
로컬 1234/1234개 · 손상 0개


## 8-1. 학습 데이터를 로컬 디스크로 복사

Drive 마운트는 네트워크 파일시스템이라 매 에폭 수백 개를 원격에서 읽는다.
런타임 로컬 디스크로 옮기면 GPU가 데이터를 기다리는 시간이 줄어든다.

런타임이 끊기면 사라지므로 세션마다 다시 실행한다. 복사에 1~2분 걸린다.

In [67]:
import shutil
import time

LOCAL_ROOT = Path("/content/data")
LOCAL_PROCESSED = LOCAL_ROOT / "processed"

started = time.time()
if not LOCAL_PROCESSED.exists():
    shutil.copytree(DRIVE_PROCESSED, LOCAL_PROCESSED)

# 매니페스트의 clip_path가 "processed/..." 라서 data_root만 바꾸면 그대로 맞는다.
TRAIN_ROOT = LOCAL_ROOT
local_count = len(list(LOCAL_PROCESSED.glob("*.npy")))
print(f"로컬 npy {local_count}개 · {time.time() - started:.0f}초")

로컬 npy 1234개 · 0초


### (45 프레임)


In [70]:
import shutil, time

LOCAL_F45 = Path("/content/data45")
target = LOCAL_F45 / "processed"      # 매니페스트가 "processed/..." 로 적으므로 이 이름이어야 함
expected = len(list(PROCESSED_F45.glob("*.npy")))

if target.exists() and len(list(target.glob("*.npy"))) != expected:
    shutil.rmtree(target)             # 개수 안 맞으면 다시 복사 (8-1 셀의 그 함정)
if not target.exists():
    started = time.time()
    shutil.copytree(PROCESSED_F45, target)
    print(f"복사 {time.time() - started:.0f}초")

TRAIN_ROOT_F45 = LOCAL_F45
print(f"로컬 npy {len(list(target.glob('*.npy')))}개 / 기대 {expected}개")

KeyboardInterrupt: 

## 9. 학습

체크포인트는 Drive에 저장되므로 런타임이 끊겨도 남는다.
학습 데이터는 `TRAIN_ROOT`(로컬 복사본)에서 읽어 I/O 대기를 줄인다.

**실험은 이 셀의 인자만 바꾸면 된다.** 저장소 코드를 고칠 필요가 없다.

- `seed` — 가중치 초기값·데이터 순서·증강을 한꺼번에 고정한다.
  같은 시드면 같은 결과가 나오므로 설정 비교의 전제가 된다
- `val_speakers` — 검증에 쓸 화자. **설정을 비교할 때는 반드시 고정한다.**
  `None`이면 화자 구성이 바뀔 때 검증 대상도 함께 바뀌어 비교가 깨진다
- `hidden_dim` / `num_layer` / `dropout` — 모델 크기와 정규화 강도
- `weight_decay` — 가중치를 작게 유지해 과적합을 억제
- `smoothing` — 최근 몇 에폭 평균으로 체크포인트를 판정할지. `1`이면 단일 에폭 최고치
- `label_smoothing` — 정답 확률을 100%로 몰지 않게 해 과신을 줄인다
- `grad_clip` — 드물게 튀는 그래디언트가 가중치를 흔드는 것을 막는다
- `ema_decay` — 가중치 이동평균으로 검증한다. 후반 진동이 완만해진다
- `augment` / `augmentation_config` — 증강 사용 여부와 강도
- `pretrained` — ImageNet 가중치로 백본을 초기화. 입력 정규화도 함께 바뀐다
- `freeze_backbone` — ResNet 층을 고정. `pretrained`와 함께 쓴다
- `amp` — bfloat16 혼합정밀도. GPU에서만 켜지고 속도가 2~3배 빨라진다
- `wandb_project` — 지정하면 실험이 웹 대시보드에 자동 기록된다

`label_smoothing` · `grad_clip` · `ema_decay`는 안정화 장치다. 셋 다 `0`을 주면
꺼진다. 체크포인트는 EMA를 켜면 평균 가중치로 저장되므로 검증 수치와 일치한다.

**시드 하나로 낸 결과는 그 자체로 성능이 아니다.** 같은 설정이라도 시드가 다르면
0.05~0.1 정도 흔들린다. 설정을 비교하거나 최종 수치를 낼 때는 시드 2~3개로
돌려 평균을 쓴다.

`pretrained=True`에 `freeze_backbone=False`면 학습률을 `3e-5`로 낮춘다. 좋은
초기값을 큰 보폭이 흐트러뜨리기 때문이다. 반대로 동결하면 움직이는 파라미터가
적어 `3e-4`까지 올려도 안정적이다.

**한 번에 하나만 바꾼다.** 두 개를 동시에 바꾸면 무엇이 효과였는지 알 수 없다.
`run_name`에 설정을 알아볼 수 있는 이름을 붙이면 나중에 비교하기 쉽다.

In [ ]:
from src.ml.preprocess.augmentation import AugmentationConfig
from src.ml.training.train import train

# 증강 강도를 조절하려면 설정을 만들어 넘긴다. None이면 기본값을 쓴다.
strong_augmentation = AugmentationConfig(
    brightness_probability=0.7,
    contrast_probability=0.7,
    rotation_probability=0.6,
    shift_probability=0.6,
    zoom_probability=0.6,
)

best_accuracy = train(
    manifest_path=manifest_path,
    data_root=TRAIN_ROOT,
    epochs=80,
    batch_size=16,
    learning_rate=2e-4,
    seed=42,  # 가중치 초기값·데이터 순서·증강을 함께 고정한다
    val_speakers=["s04"],  # 설정 비교 시 고정. None이면 seed로 무작위 선택
    checkpoint_path=DRIVE_CHECKPOINTS / "best.pt",
    num_workers= 8,
    amp=True,
    hidden_dim=300,
    num_layer=2,
    dropout=0.3,
    weight_decay=0.01,
    smoothing=3,
    label_smoothing=0.1,  # 0이면 끔. 정답에 대한 과신을 줄인다
    grad_clip=1.0,  # 0이면 끔. 튀는 그래디언트를 잘라낸다
    ema_decay=0.998,  # 0이면 끔. 가중치 이동평균으로 검증한다
    augment=True,
    augmentation_config=None,  # strong_augmentation 으로 바꿔 강도 실험
    pretrained=False,  # True면 ImageNet 가중치 + ImageNet 입력 정규화
    freeze_backbone=False,  # pretrained와 함께 켜면 ResNet 층을 고정한다
    wandb_project="lipreading",
    run_name="baseline_s04",
)

### (45 프레임)


In [ ]:
from src.ml.training.train import train

SEEDS = [42, 1, 7]
results45 = {}

for seed in SEEDS:
    print(f"\n{'='*16} s06 · seed {seed} · {FRAMES}프레임 {'='*16}")
    results45[seed] = train(
        manifest_path=manifest_f45,
        data_root=TRAIN_ROOT_F45,
        epochs=80,
        batch_size=16,
        learning_rate=2e-4,
        seed=seed,
        val_speakers=["s06"],
        checkpoint_path=DRIVE_CHECKPOINTS / f"f45_s06_seed{seed}.pt",
        num_workers=8,
        amp=True,
        ema_decay=0.998,
        hidden_dim=300,
        num_layer=2,
        dropout=0.3,
        smoothing=3,
        wandb_project="lipreading",
        run_name=f"f45_s06_seed{seed}",
    )

values = [results45[s] for s in SEEDS]
print(f"\n{'='*50}")
print(f"45프레임   {[f'{v:.3f}' for v in values]}")
print(f"           평균 {sum(values)/len(values):.3f} · 폭 {max(values)-min(values):.3f}")
print(f"30프레임   평균 0.754 · 폭 0.051")
print(f"판정선     0.804     ← 넘으면 채택")

### (60 프레임)

In [71]:
# ═══ 60프레임 8화자 교차검증 · seed 42 ═══
from src.ml.training.train import train

SEEDS = [42, 1, 7]
speakers = sorted({row["speaker_id"] for row in rows})
print(f"화자 {speakers} · 시드 {SEEDS} · 60프레임\n")

results60cv = {}
for speaker in speakers:
    for seed in SEEDS:
        print(f"\n{'='*16} {speaker} · seed {seed} · 60프레임 {'='*16}")
        results60cv[(speaker, seed)] = train(
            manifest_path=manifest_f60,          # ← 60프레임
            data_root=TRAIN_ROOT_F60,
            epochs=80,
            batch_size=16,
            learning_rate=2e-4,
            seed=seed,
            val_speakers=[speaker],
            checkpoint_path=DRIVE_CHECKPOINTS / f"cv60_{speaker}_seed{seed}.pt",
            num_workers=8,
            amp=True,
            ema_decay=0.998,
            hidden_dim=300,
            num_layer=2,
            dropout=0.3,
            smoothing=3,
            wandb_project="lipreading",
            run_name=f"cv60_{speaker}_seed{seed}",
        )

BASE30 = {"s01": 0.656, "s03": 0.446, "s04": 0.780, "s05": 0.307,
          "s06": 0.732, "s07": 0.351, "s08": 0.556, "s09": 0.393}

print(f"\n{'='*56}")
print(f"{'화자':<6}{'30프레임':>10}{'60프레임':>10}{'차이':>10}")
print("-" * 40)
per = {}
for sp in speakers:
    v = sum(results60cv[(sp, s)] for s in SEEDS) / len(SEEDS)
    per[sp] = v
    print(f"{sp:<6}{BASE30[sp]:>10.3f}{v:>10.3f}{v - BASE30[sp]:>+10.3f}")

new, old = sum(per.values()) / len(per), sum(BASE30.values()) / len(BASE30)
print("-" * 40)
print(f"{'평균':<6}{old:>10.3f}{new:>10.3f}{new - old:>+10.3f}")
print(f"\n화자 간 편차   30프레임 {max(BASE30.values())-min(BASE30.values()):.3f}"
      f" · 60프레임 {max(per.values())-min(per.values()):.3f}")

화자 ['s01', 's03', 's04', 's05', 's06', 's07', 's08', 's09'] · 시드 [42, 1, 7] · 60프레임


================ s01 · seed 42 · 60프레임 ================


장치: cuda | 클래스: 15개
학습 1077개 · 검증 157개 클립 | 검증 화자 ['s01']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7305 acc 0.077 | val loss 2.7074 acc 0.064 avg 0.064 | lr 2.00e-04
[  2/80] train loss 2.4088 acc 0.216 | val loss 2.7088 acc 0.064 avg 0.064 | lr 2.00e-04
[  3/80] train loss 2.0713 acc 0.357 | val loss 2.7214 acc 0.070 avg 0.066 | lr 1.99e-04
[  4/80] train loss 1.7052 acc 0.562 | val loss 2.7519 acc 0.070 avg 0.068 | lr 1.99e-04
[  5/80] train loss 1.3745 acc 0.733 | val loss 2.8142 acc 0.070 avg 0.070 | lr 1.98e-04
[  6/80] train loss 1.1309 acc 0.834 | val loss 2.8924 acc 0.076 avg 0.072 | lr 1.97e-04
[  7/80] train loss 0.9667 acc 0.872 | val loss 2.9975 acc 0.076 avg 0.074 | lr 1.96e-04
[  8/80] train loss 0.9082 acc 0.893 | val loss 3.1010 acc 0.076 avg 0.076 | lr 1.95e-04
[  9/80] train loss 0.8190 acc 0.926 | val loss

lr,█████▇▇▇▇▇▇▆▆▆▆▅▅▅▅▅▅▄▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁
train/acc,▁▅▇▇████████████████████████████████████
train/loss,█▇▅▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▁▁▁▁▁▂▃▃▃▄▅▆▆▇▇▇████████████████████
val/acc_smoothed,▁▁▁▁▁▁▂▂▂▃▄▅▅▅▆▆▇▇▇▇████████████████████
val/loss,▆▆▇██▇▆▆▅▅▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_acc,0.71338
best_val_acc_smoothed,0.71125
lr,0
train/acc,1
train/loss,0.56217



================ s01 · seed 1 · 60프레임 ================


장치: cuda | 클래스: 15개
학습 1077개 · 검증 157개 클립 | 검증 화자 ['s01']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7116 acc 0.085 | val loss 2.7082 acc 0.070 avg 0.070 | lr 2.00e-04
[  2/80] train loss 2.4274 acc 0.186 | val loss 2.7153 acc 0.070 avg 0.070 | lr 2.00e-04
[  3/80] train loss 2.1758 acc 0.316 | val loss 2.7330 acc 0.070 avg 0.070 | lr 1.99e-04
[  4/80] train loss 1.8787 acc 0.474 | val loss 2.7806 acc 0.070 avg 0.070 | lr 1.99e-04
[  5/80] train loss 1.5288 acc 0.659 | val loss 2.8650 acc 0.076 avg 0.072 | lr 1.98e-04
[  6/80] train loss 1.2726 acc 0.746 | val loss 2.9839 acc 0.076 avg 0.074 | lr 1.97e-04
[  7/80] train loss 1.0904 acc 0.824 | val loss 3.1424 acc 0.076 avg 0.076 | lr 1.96e-04
[  8/80] train loss 0.9293 acc 0.892 | val loss 3.3034 acc 0.076 avg 0.076 | lr 1.95e-04
[  9/80] train loss 0.8509 acc 0.916 | val loss

...

lr,████████▇▇▇▇▇▇▇▇▆▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁
train/acc,▁▆▇▇████████████████████████████████████
train/loss,█▅▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▁▂▄▆▆▇▇█▇▇█▇▇▇▇▇▇██████████████▇██▇▇
val/acc_smoothed,▁▁▁▁▁▁▁▄▅▆█████▇▇▇█████████████████████▇
val/loss,▅▅▆▇█▆▅▃▂▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▃▃
best_val_acc,0.43333
best_val_acc_smoothed,0.44
lr,0
train/acc,1
train/loss,0.56233



================ s09 · seed 7 · 60프레임 ================


장치: cuda | 클래스: 15개
학습 1084개 · 검증 150개 클립 | 검증 화자 ['s09']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7425 acc 0.080 | val loss 2.7081 acc 0.067 avg 0.067 | lr 2.00e-04
[  2/80] train loss 2.4908 acc 0.178 | val loss 2.7109 acc 0.067 avg 0.067 | lr 2.00e-04
[  3/80] train loss 2.1988 acc 0.319 | val loss 2.7195 acc 0.067 avg 0.067 | lr 1.99e-04
[  4/80] train loss 1.8517 acc 0.485 | val loss 2.7489 acc 0.067 avg 0.067 | lr 1.99e-04
[  5/80] train loss 1.5201 acc 0.653 | val loss 2.8305 acc 0.067 avg 0.067 | lr 1.98e-04
[  6/80] train loss 1.2203 acc 0.786 | val loss 2.9927 acc 0.067 avg 0.067 | lr 1.97e-04
[  7/80] train loss 1.0071 acc 0.864 | val loss 3.2071 acc 0.067 avg 0.067 | lr 1.96e-04
[  8/80] train loss 0.8873 acc 0.911 | val loss 3.4037 acc 0.067 avg 0.067 | lr 1.95e-04
[  9/80] train loss 0.8215 acc 0.927 | val loss

lr,████████▇▇▇▇▇▇▆▆▆▆▅▅▄▄▄▄▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁
train/acc,▁▃▅▆▇▇██████████████████████████████████
train/loss,█▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▁▄▅▆▇▇▇▇▇██▇▇▇█▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
val/acc_smoothed,▁▁▁▁▁▄▅▅▇▇▇███████▇█▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
val/loss,▅▅▅▆█▆▅▄▃▂▂▂▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃
best_val_acc,0.43333
best_val_acc_smoothed,0.44
lr,0
train/acc,1
train/loss,0.56228



화자         30프레임     60프레임        차이
----------------------------------------
s01        0.656     0.741    +0.085
s03        0.446     0.561    +0.115
s04        0.780     0.769    -0.011
s05        0.307     0.231    -0.076
s06        0.732     0.783    +0.051
s07        0.351     0.312    -0.039
s08        0.556     0.667    +0.111
s09        0.393     0.424    +0.031
----------------------------------------
평균         0.528     0.561    +0.033

화자 간 편차   30프레임 0.473 · 60프레임 0.552


## 9-1. 교차검증

검증 화자 한 명으로 재면 그 사람의 난이도에 결과가 좌우된다. 화자를 바꿔가며
전부 한 번씩 검증으로 쓰고 평균을 내면 화자 편차에 흔들리지 않는 수치가 나온다.

**시드도 함께 반복해야 한다.** 같은 설정이라도 시드가 다르면 0.05~0.1 흔들리는데,
이 폭이 화자 간 차이와 비슷해서 한 번씩만 돌리면 순위를 신뢰할 수 없다.

`SEEDS`를 늘릴수록 신뢰도가 올라가지만 학습 횟수가 화자 수만큼 곱해진다.
경향만 볼 때는 시드 하나로, 발표에 쓸 최종 수치는 셋으로 돌린다.

In [ ]:
SEEDS = [42]  # [42, 1, 7] 로 늘리면 화자마다 여러 번 돌려 편차까지 본다

speakers = sorted({row["speaker_id"] for row in rows})
print(f"화자 {speakers} · 시드 {SEEDS}")

results = {}
for speaker in speakers:
    for seed in SEEDS:
        print(f"\n{'=' * 18} {speaker} · seed {seed} {'=' * 18}")
        results[(speaker, seed)] = train(
            manifest_path=manifest_path,
            data_root=TRAIN_ROOT,
            epochs=80,
            batch_size=16,
            learning_rate=2e-4,
            seed=seed,
            val_speakers=[speaker],
            checkpoint_path=DRIVE_CHECKPOINTS / f"cv_{speaker}_seed{seed}.pt",
            num_workers=8,
            amp=True,
            ema_decay=0.998,
            hidden_dim=300,
            num_layer=2,
            dropout=0.3,
            smoothing=3,
            wandb_project="lipreading",
            run_name=f"cv_{speaker}_seed{seed}",
        )

print(f"\n{'=' * 50}")
per_speaker = {}
for speaker in speakers:
    values = [results[(speaker, s)] for s in SEEDS]
    per_speaker[speaker] = sum(values) / len(values)
    detail = " ".join(f"{v:.3f}" for v in values)
    spread = f" (폭 {max(values) - min(values):.3f})" if len(values) > 1 else ""
    print(f"  {speaker}: {per_speaker[speaker]:.3f}   [{detail}]{spread}")

overall = sum(per_speaker.values()) / len(per_speaker)
gap = max(per_speaker.values()) - min(per_speaker.values())
print(f"\n전체 평균 {overall:.3f} · 화자 간 편차 {gap:.3f}")

## 10. 체크포인트 확인

In [ ]:
checkpoint = torch.load(DRIVE_CHECKPOINTS / "best.pt", map_location="cpu")

print(f"에폭 {checkpoint['epoch']}")
print(f"클래스 {checkpoint['num_classes']}개")
print(f"검증 정확도 {checkpoint['val_accuracy']:.3f}")
print(f"최근 평균 {checkpoint['smoothed_accuracy']:.3f}")
print(
    f"모델 hidden {checkpoint['hidden_dim']} · "
    f"layer {checkpoint['num_layer']} · dropout {checkpoint['dropout']}"
)

In [ ]:
from pathlib import Path
import shutil, numpy as np

local = Path(TRAIN_ROOT) / "processed"

drive_names = {p.name for p in DRIVE_PROCESSED.glob("*.npy")}
local_names = {p.name for p in local.glob("*.npy")}
missing = drive_names - local_names
broken  = {p.name for p in local.glob("*.npy") if p.stat().st_size == 0}
fix = missing | broken

print(f"누락 {len(missing)} · 0바이트 {len(broken)} · 복구 대상 {len(fix)}개")
for name in sorted(fix):
    src = DRIVE_PROCESSED / name
    shutil.copy2(src, local / name)
    print(f"  {name}  ({src.stat().st_size:,}바이트)")

# 검증
bad = []
for p in local.glob("*.npy"):
    try:
        if p.stat().st_size == 0:
            raise ValueError
        np.load(p, mmap_mode="r")
    except Exception:
        bad.append(p.name)

print(f"\n로컬 {len(list(local.glob('*.npy')))}개 / Drive {len(drive_names)}개 · 손상 {len(bad)}개")
assert len(bad) == 0 and len(local_names | fix) == len(drive_names), "아직 안 맞습니다"

In [ ]:
import torch, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from src.ml.models.lip_reading_model import LipReadingModel
from src.ml.training.dataset import LipReadingDataset
from src.ml.training.train import split_by_speaker

SPEAKERS = ["s01", "s03", "s04", "s05", "s06", "s07", "s08", "s09"]
TAU_MAIN = 1.0          # 사전 선택값. 이걸로 판정한다

def probs_fold(speaker):
    ds = LipReadingDataset(manifest_path, TRAIN_ROOT)
    _, vi, _ = split_by_speaker(ds, val_speakers=[speaker])
    loader = DataLoader(Subset(ds, vi), batch_size=16, num_workers=2)
    ck = torch.load(DRIVE_CHECKPOINTS / f"cv_{speaker}_seed42.pt", map_location="cuda")
    m = LipReadingModel(num_classes=ck["num_classes"], hidden_dim=ck["hidden_dim"],
                        num_layer=ck["num_layer"], dropout=ck["dropout"]).cuda()
    m.load_state_dict(ck["model_state"]); m.eval()
    P, Y = [], []
    with torch.no_grad():
        for x, y in loader:
            with torch.autocast("cuda", dtype=torch.bfloat16):
                o = m(x.cuda())
            P.append(F.softmax(o.float(), 1).cpu()); Y.append(y)
    return torch.cat(P), torch.cat(Y), sorted({r["label_text"] for r in ds.rows})

fold = {}
for sp in SPEAKERS:
    fold[sp] = probs_fold(sp)
    print(f"{sp} 완료")

texts = fold["s01"][2]
C = len(texts)
total = sum(len(fold[s][1]) for s in SPEAKERS)

# ── 1. 예측 분포가 얼마나 치우쳤나 ──
pred_n, true_n = torch.zeros(C), torch.zeros(C)
for P, Y, _ in fold.values():
    pred_n += torch.bincount(P.argmax(1), minlength=C).float()
    true_n += torch.bincount(Y, minlength=C).float()

print(f"\n{'문구':<16}{'정답':>6}{'예측':>6}{'배율':>7}")
print("-" * 37)
for i in torch.argsort(pred_n / true_n, descending=True):
    print(f"{texts[i]:<16}{int(true_n[i]):>6}{int(pred_n[i]):>6}{pred_n[i] / true_n[i]:>7.2f}")

# ── 2. 보정 ──
def freq(speakers):
    n = torch.zeros(C)
    for s in speakers:
        n += torch.bincount(fold[s][0].argmax(1), minlength=C).float()
    return (n / n.sum()).clamp(min=1e-6)

base = sum((fold[s][0].argmax(1) == fold[s][1]).sum().item() for s in SPEAKERS) / total

print(f"\n{'τ':>5}{'상한(자기 화자)':>16}{'정직(타화자 추정)':>18}")
print("-" * 40)
for tau in (0.0, 0.25, 0.5, 0.75, 1.0):
    hi = ho = 0
    for sp in SPEAKERS:
        P, Y, _ = fold[sp]
        others = [s for s in SPEAKERS if s != sp]
        hi += ((P / freq([sp]) ** tau).argmax(1) == Y).sum().item()
        ho += ((P / freq(others) ** tau).argmax(1) == Y).sum().item()
    mark = "  ← 판정" if tau == TAU_MAIN else ""
    print(f"{tau:>5.2f}{hi / total:>16.3f}{ho / total:>18.3f}{mark}")

print(f"\n보정 없음  {base:.3f}")

# ── 3. τ=1 정직 버전, 화자별 ──
print(f"\n{'화자':<6}{'보정전':>8}{'보정후':>8}{'차이':>8}")
print("-" * 32)
for sp in SPEAKERS:
    P, Y, _ = fold[sp]
    others = [s for s in SPEAKERS if s != sp]
    b = (P.argmax(1) == Y).float().mean().item()
    a = ((P / freq(others) ** TAU_MAIN).argmax(1) == Y).float().mean().item()
    print(f"{sp:<6}{b:>8.3f}{a:>8.3f}{a - b:>+8.3f}")

In [ ]:
import collections

for target_name in ["도와주세요", "자세바꿔주세요", "숨쉬기힘들어요"]:
    t = texts.index(target_name)
    print(f"\n=== {target_name} ===")
    print(f"{'화자':<6}{'정답':>6}{'맞춤':>6}{'재현율':>8}  주요 오답")
    for sp in SPEAKERS:
        P, Y, _ = fold[sp]
        mask = Y == t
        n = int(mask.sum())
        if n == 0:
            continue
        pred = P[mask].argmax(1)
        hit = int((pred == t).sum())
        wrong = collections.Counter(texts[int(i)] for i in pred if int(i) != t)
        top = " · ".join(f"{a}×{c}" for a, c in wrong.most_common(2))
        print(f"{sp:<6}{n:>6}{hit:>6}{hit / n:>8.2f}  {top}")


In [ ]:
import numpy as np, csv, collections
from pathlib import Path

with open(manifest_path, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

acc = collections.defaultdict(list)
for r in rows:
    a = np.load(Path(TRAIN_ROOT) / r["clip_path"])[:, :, :, 0].astype(np.float32)
    motion = np.abs(np.diff(a, axis=0)).mean()
    dup = np.mean([np.array_equal(a[i], a[i + 1]) for i in range(len(a) - 1)])
    acc[r["label_text"]].append((motion, dup))

print(f"{'문구':<16}{'움직임':>8}{'중복률':>8}{'클립':>6}")
print("-" * 40)
for m, ph, d, n in sorted((np.mean([x[0] for x in v]), ph,
                           np.mean([x[1] for x in v]), len(v))
                          for ph, v in acc.items()):
    print(f"{ph:<16}{m:>8.2f}{d:>8.2f}{n:>6}")

# 증강


In [ ]:
import importlib, sys
from src.ml.preprocess.augmentation import pipeline

fresh = importlib.reload(pipeline)                    # 소스에서 원본을 새로 읽음
patched = sys.modules["src.ml.training.train"].VideoAugmentation
patched.__call__ = fresh.VideoAugmentation.__call__   # 학습 코드가 쥔 클래스에 되돌림
print("복구:", patched.__call__.__qualname__)          # VideoAugmentation.__call__ 이면 정상

In [ ]:
# ═══ 시간축 증강 실험 · 이 셀 하나만 실행 (재실행 안전) ═══
from pathlib import Path
import numpy as np
from src.ml.preprocess.augmentation.pipeline import VideoAugmentation
from src.ml.training.train import train

DRIVE_ROOT        = globals().get("DRIVE_ROOT", Path("/content/drive/MyDrive/hanium-lipreading"))
DRIVE_CHECKPOINTS = globals().get("DRIVE_CHECKPOINTS", DRIVE_ROOT / "checkpoints")
manifest_path     = globals().get("manifest_path", DRIVE_ROOT / "manifest.csv")
TRAIN_ROOT        = globals().get("TRAIN_ROOT",
                    Path("/content/data") if Path("/content/data/processed").exists() else DRIVE_ROOT)
assert manifest_path.exists(), f"매니페스트 없음: {manifest_path}"

TIME_CROP_PROB = 0.5
TIME_CROP_MIN  = 0.75
SEEDS = [42, 1, 7]

# 원본을 클래스 속성에 한 번만 보관 → 몇 번 실행해도 진짜 원본이 유지된다
if not getattr(VideoAugmentation, "_taug_patched", False):
    VideoAugmentation._taug_orig = VideoAugmentation.__call__
    VideoAugmentation._taug_patched = True
ORIG = VideoAugmentation._taug_orig
assert ORIG.__qualname__ == "VideoAugmentation.__call__", f"원본이 아님: {ORIG.__qualname__}"

def call_with_time_aug(self, clip, return_details=False):
    if return_details:
        return ORIG(self, clip, True)
    frames = ORIG(self, clip, False)
    if self.rng.random() < TIME_CROP_PROB:
        T = len(frames)
        keep = int(T * self.rng.uniform(TIME_CROP_MIN, 1.0))
        if 2 <= keep < T:
            start = int(self.rng.integers(0, T - keep + 1))
            idx = np.linspace(start, start + keep - 1, T).round().astype(int)
            frames = frames[idx]
    return frames

VideoAugmentation.__call__ = call_with_time_aug
print(f"시간축 증강 적용 · 확률 {TIME_CROP_PROB} · 크롭 하한 {TIME_CROP_MIN}")
print(f"데이터 {TRAIN_ROOT}\n")

results = {}
try:
    for seed in SEEDS:
        print(f"\n{'='*16} s06 · seed {seed} · 시간축 증강 {'='*16}")
        results[seed] = train(
            manifest_path=manifest_path, data_root=TRAIN_ROOT,
            epochs=80, batch_size=16, learning_rate=2e-4,
            seed=seed, val_speakers=["s06"],
            checkpoint_path=DRIVE_CHECKPOINTS / f"taug_s06_seed{seed}.pt",
            num_workers=8, amp=True, ema_decay=0.998,
            hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
            wandb_project="lipreading", run_name=f"taug_s06_seed{seed}",
        )
finally:
    VideoAugmentation.__call__ = ORIG
    print("\n[증강 원상복구 완료]")

v = [results[s] for s in SEEDS if s in results]
print(f"\n{'='*56}")
if len(v) == len(SEEDS):
    print(f"시간축 증강   {[f'{x:.3f}' for x in v]}   평균 {sum(v)/3:.3f} · 폭 {max(v)-min(v):.3f}")
else:
    print(f"완료 {len(v)}/{len(SEEDS)}시드   {[f'{x:.3f}' for x in v]}")
print(f"기준선(30)   ['0.732', '0.783', '0.745']   평균 0.754 · 폭 0.051")
print(f"채택선       0.805")

#오디오추출


In [5]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, shutil, time
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/hanium-lipreading")
DRIVE_CHECKPOINTS = DRIVE_ROOT / "checkpoints"
PROCESSED_F60 = DRIVE_ROOT / "processed_f60"

n_drive = len(list(PROCESSED_F60.glob("*.npy")))
print("Drive processed_f60:", n_drive, "개")
assert n_drive == 1234, "processed_f60이 1234개가 아니다 - 전처리 이력 확인 필요"

REPO_DIR = Path("/content/hanium-lipreading")
os.chdir("/content")
if (REPO_DIR / ".git").exists():
    !cd {REPO_DIR\} && git fetch origin && git checkout develop && git pull
else:
    !git clone -b develop https://github.com/HumanRhoid/hanium-lipreading.git {REPO_DIR\}
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
!pip install --quiet wandb

from scripts.build_manifest import build
manifest_f60 = DRIVE_ROOT / "manifest_f60.csv"
build(processed_dir=PROCESSED_F60, manifest_path=manifest_f60)

TRAIN_ROOT_F60 = Path("/content/data_f60")
LOCAL_F60 = TRAIN_ROOT_F60 / "processed"
started = time.time()
have = len(list(LOCAL_F60.glob("*.npy"))) if LOCAL_F60.exists() else 0
if have != n_drive:
    shutil.rmtree(TRAIN_ROOT_F60, ignore_errors=True)
    shutil.copytree(PROCESSED_F60, LOCAL_F60)

n_local = len(list(LOCAL_F60.glob("*.npy")))
bad = [p.name for p in LOCAL_F60.glob("*.npy") if p.stat().st_size == 0]
print("로컬", n_local, "개 ·", round(time.time() - started), "초 · 0바이트", len(bad), "개")
assert n_local == n_drive and not bad, "로컬 복사 불완전"
# end$0

Mounted at /content/drive
Drive processed_f60: 1234 개
Cloning into '{REPO_DIR}'...
remote: Enumerating objects: 667, done.
remote: Counting objects: 100% (226/226), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 667 (delta 96), reused 110 (delta 62), pack-reused 441 (from 1)
Receiving objects: 100% (667/667), 669.11 KiB | 5.72 MiB/s, done.
Resolving deltas: 100% (300/300), done.


FileNotFoundError: [Errno 2] No such file or directory: '/content/hanium-lipreading'

In [6]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, shutil, time, subprocess
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/hanium-lipreading")
DRIVE_CHECKPOINTS = DRIVE_ROOT / "checkpoints"
PROCESSED_F60 = DRIVE_ROOT / "processed_f60"

n_drive = len(list(PROCESSED_F60.glob("*.npy")))
print("Drive processed_f60:", n_drive, "개")
assert n_drive == 1234, "processed_f60이 1234개가 아니다"

os.chdir("/content")
for junk in Path("/content").glob("*REPO_DIR*"):
    shutil.rmtree(junk, ignore_errors=True)
    print("잘못 만들어진 폴더 삭제:", junk.name)

REPO = "/content/hanium-lipreading"
REPO_DIR = Path(REPO)
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", REPO, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "develop"], check=True)
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    shutil.rmtree(REPO, ignore_errors=True)
    subprocess.run(["git", "clone", "-b", "develop",
                    "https://github.com/HumanRhoid/hanium-lipreading.git", REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "wandb"], check=True)

from scripts.build_manifest import build
manifest_f60 = DRIVE_ROOT / "manifest_f60.csv"
build(processed_dir=PROCESSED_F60, manifest_path=manifest_f60)

TRAIN_ROOT_F60 = Path("/content/data_f60")
LOCAL_F60 = TRAIN_ROOT_F60 / "processed"
started = time.time()
have = len(list(LOCAL_F60.glob("*.npy"))) if LOCAL_F60.exists() else 0
if have != n_drive:
    shutil.rmtree(TRAIN_ROOT_F60, ignore_errors=True)
    shutil.copytree(PROCESSED_F60, LOCAL_F60)

n_local = len(list(LOCAL_F60.glob("*.npy")))
bad = [p.name for p in LOCAL_F60.glob("*.npy") if p.stat().st_size == 0]
print("로컬", n_local, "개 ·", round(time.time() - started), "초 · 0바이트", len(bad), "개")
assert n_local == n_drive and not bad, "로컬 복사 불완전"
# end$0

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive processed_f60: 1234 개
잘못 만들어진 폴더 삭제: {REPO_DIR}
매니페스트 생성: /content/drive/MyDrive/hanium-lipreading/manifest_f60.csv
  클립 1234개 · 문구 15개 · 화자 8명
  라벨 매핑: 0=가래가있어요, 1=간호사불러주세요, 2=더워요, 3=도와주세요, 4=물주세요, 5=배고파요, 6=보호자불러주세요, 7=숨쉬기힘들어요, 8=아파요, 9=어지러워요, 10=자세바꿔주세요, 11=진통제주세요, 12=추워요, 13=토할거같아요, 14=화장실가고싶어요
로컬 1234 개 · 38 초 · 0바이트 0 개


In [7]:
from src.ml.training.train import train
from src.ml.preprocess.augmentation import VideoAugmentation

print("call:", VideoAugmentation.__call__.__qualname__)
assert VideoAugmentation.__call__.__qualname__ == "VideoAugmentation.__call__", "패치가 걸려 있다"

acc = train(
    manifest_path=manifest_f60, data_root=TRAIN_ROOT_F60,
    epochs=80, batch_size=16, learning_rate=2e-4, seed=42,
    val_speakers=["s06"],
    checkpoint_path=DRIVE_CHECKPOINTS / "repro_s06_seed42.pt",
    num_workers=8, amp=True, ema_decay=0.998,
    hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
    wandb_project="lipreading", run_name="repro_s06_seed42")

print("")
print("재현 결과 ", round(acc, 3))
print("  08-19b   0.809")
print("  08-20    0.739")
if abs(acc - 0.739) < abs(acc - 0.809):
    print("  판정: 08-19b가 오염됨")
else:
    print("  판정: 08-20 8-fold가 오염됨")
# end$0

call: VideoAugmentation.__call__


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


...

wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ssanta011205 (ssanta011205-seokyeong-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


장치: cuda | 클래스: 15개
학습 1077개 · 검증 157개 클립 | 검증 화자 ['s06']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7311 acc 0.081 | val loss 2.7094 acc 0.057 avg 0.057 | lr 2.00e-04
[  2/80] train loss 2.4679 acc 0.163 | val loss 2.7087 acc 0.057 avg 0.057 | lr 2.00e-04
[  3/80] train loss 2.2077 acc 0.300 | val loss 2.7130 acc 0.057 avg 0.057 | lr 1.99e-04
[  4/80] train loss 1.9030 acc 0.436 | val loss 2.7297 acc 0.070 avg 0.062 | lr 1.99e-04
[  5/80] train loss 1.5475 acc 0.633 | val loss 2.7695 acc 0.070 avg 0.066 | lr 1.98e-04
[  6/80] train loss 1.2625 acc 0.765 | val loss 2.8498 acc 0.070 avg 0.070 | lr 1.97e-04
[  7/80] train loss 1.0336 acc 0.861 | val loss 2.9844 acc 0.070 avg 0.070 | lr 1.96e-04
[  8/80] train loss 0.9301 acc 0.892 | val loss 3.0874 acc 0.070 avg 0.070 | lr 1.95e-04
[  9/80] train loss 0.8452 acc 0.926 | val loss

lr,██████▇▇▇▇▇▇▇▆▆▅▅▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
train/acc,▁▃▇▇▇███████████████████████████████████
train/loss,█▅▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▂▄▆▇▇▇▇▇▇▇▇▇████████████████████████
val/acc_smoothed,▁▁▁▁▁▁▁▂▂▂▂▃▄▆▇▇▇▇▇▇▇▇▇█████████████████
val/loss,▆▆▇▇▇█▆▆▅▄▂▂▂▂▁▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_acc,0.82166
best_val_acc_smoothed,0.82378
lr,0
train/acc,1
train/loss,0.56228



재현 결과  0.822
  08-19b   0.809
  08-20    0.739
  판정: 08-20 8-fold가 오염됨


In [8]:
from src.ml.training.train import train

base80 = [0.713, 0.815, 0.694]
res = []
for sd in (42, 1, 7):
    print("")
    print("=== s01 · seed", sd, "· 120에폭 ===")
    a = train(
        manifest_path=manifest_f60, data_root=TRAIN_ROOT_F60,
        epochs=120, batch_size=16, learning_rate=2e-4, seed=sd,
        val_speakers=["s01"],
        checkpoint_path=DRIVE_CHECKPOINTS / ("ep120_s01_seed" + str(sd) + ".pt"),
        num_workers=8, amp=True, ema_decay=0.998,
        hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
        wandb_project="lipreading", run_name="ep120_s01_seed" + str(sd))
    res.append(round(a, 3))
    print("누적:", res)

m80 = sum(base80) / 3
m120 = sum(res) / 3
print("")
print("80에폭  ", base80, "평균", round(m80, 3), "폭", round(max(base80) - min(base80), 3))
print("120에폭 ", res, "평균", round(m120, 3), "폭", round(max(res) - min(res), 3))
print("차이", round(m120 - m80, 3))
# end


=== s01 · seed 42 · 120에폭 ===


장치: cuda | 클래스: 15개
학습 1077개 · 검증 157개 클립 | 검증 화자 ['s01']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/120] train loss 2.7311 acc 0.077 | val loss 2.7092 acc 0.064 avg 0.064 | lr 2.00e-04
[  2/120] train loss 2.3757 acc 0.230 | val loss 2.7076 acc 0.064 avg 0.064 | lr 2.00e-04
[  3/120] train loss 2.0487 acc 0.370 | val loss 2.7165 acc 0.076 avg 0.068 | lr 2.00e-04
[  4/120] train loss 1.7415 acc 0.516 | val loss 2.7574 acc 0.076 avg 0.072 | lr 1.99e-04
[  5/120] train loss 1.4397 acc 0.683 | val loss 2.8415 acc 0.076 avg 0.076 | lr 1.99e-04
[  6/120] train loss 1.1773 acc 0.802 | val loss 2.9592 acc 0.076 avg 0.076 | lr 1.99e-04
[  7/120] train loss 0.9981 acc 0.860 | val loss 3.0713 acc 0.076 avg 0.076 | lr 1.98e-04
[  8/120] train loss 0.9329 acc 0.882 | val loss 3.1736 acc 0.076 avg 0.076 | lr 1.98e-04
[  9/120] train loss 0.8390 acc 0.920 |

lr,█████▇▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▄▃▃▃▃▃▂▂▂▁▁▁▁▁▁▁
train/acc,▁▄▆▇████████████████████████████████████
train/loss,█▇▅▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▁▃▄▅▅▆▆▇▇▇▇▇▇▇▇▇▇███████████████▇▇▇▇
val/acc_smoothed,▁▁▁▁▁▁▄▅▅▆▇▇▇▇▇▇▇▇▇█████████████▇▇▇▇▇▇▇▇
val/loss,▅▅▆██▄▄▃▃▃▂▂▂▁▁▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_acc,0.67516
best_val_acc_smoothed,0.67728
lr,0
train/acc,1
train/loss,0.56056


누적: [0.675]

=== s01 · seed 1 · 120에폭 ===


장치: cuda | 클래스: 15개
학습 1077개 · 검증 157개 클립 | 검증 화자 ['s01']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/120] train loss 2.7189 acc 0.091 | val loss 2.7085 acc 0.070 avg 0.070 | lr 2.00e-04
[  2/120] train loss 2.4383 acc 0.172 | val loss 2.7167 acc 0.070 avg 0.070 | lr 2.00e-04
[  3/120] train loss 2.1804 acc 0.304 | val loss 2.7342 acc 0.070 avg 0.070 | lr 2.00e-04
[  4/120] train loss 1.9059 acc 0.452 | val loss 2.7895 acc 0.070 avg 0.070 | lr 1.99e-04
[  5/120] train loss 1.5647 acc 0.629 | val loss 2.8828 acc 0.070 avg 0.070 | lr 1.99e-04
[  6/120] train loss 1.3121 acc 0.739 | val loss 3.0216 acc 0.076 avg 0.072 | lr 1.99e-04
[  7/120] train loss 1.1002 acc 0.811 | val loss 3.1946 acc 0.076 avg 0.074 | lr 1.98e-04
[  8/120] train loss 0.9360 acc 0.901 | val loss 3.3956 acc 0.076 avg 0.076 | lr 1.98e-04
[  9/120] train loss 0.8656 acc 0.914 |

lr,█████████▇▇▇▇▇▇▆▆▆▅▅▄▄▄▄▄▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁
train/acc,▁▃▆▇▇███████████████████████████████████
train/loss,█▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▁▃▄▅▆▆▇▇▇▇▇█████████████████████████
val/acc_smoothed,▁▁▁▁▁▁▁▃▄▄▅▆▇▇▇▇▇███████████████████████
val/loss,▆▆▆▇█▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_acc,0.87261
best_val_acc_smoothed,0.87261
lr,0
train/acc,1
train/loss,0.56074


누적: [0.675, 0.873]

=== s01 · seed 7 · 120에폭 ===


장치: cuda | 클래스: 15개
학습 1077개 · 검증 157개 클립 | 검증 화자 ['s01']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/120] train loss 2.7338 acc 0.084 | val loss 2.7084 acc 0.070 avg 0.070 | lr 2.00e-04
[  2/120] train loss 2.4892 acc 0.186 | val loss 2.7115 acc 0.070 avg 0.070 | lr 2.00e-04
[  3/120] train loss 2.2075 acc 0.303 | val loss 2.7272 acc 0.070 avg 0.070 | lr 2.00e-04
[  4/120] train loss 1.8549 acc 0.484 | val loss 2.7670 acc 0.076 avg 0.072 | lr 1.99e-04
[  5/120] train loss 1.5192 acc 0.648 | val loss 2.8400 acc 0.076 avg 0.074 | lr 1.99e-04
[  6/120] train loss 1.2702 acc 0.755 | val loss 2.9266 acc 0.070 avg 0.074 | lr 1.99e-04
[  7/120] train loss 1.0996 acc 0.818 | val loss 3.0589 acc 0.070 avg 0.072 | lr 1.98e-04
[  8/120] train loss 0.9223 acc 0.894 | val loss 3.1167 acc 0.096 avg 0.079 | lr 1.98e-04
[  9/120] train loss 0.8817 acc 0.916 |

lr,██████▇▇▇▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▃▃▂▂▂▂▁▁▁▁▁▁
train/acc,▁▂▆▆▇███████████████████████████████████
train/loss,█▇▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▂▃▅▅▆▆▆▇▇▇▇▇▇█▇▇██████▇▇▇▇█████▇▇▇▇▇▇
val/acc_smoothed,▁▁▁▁▁▅▆▆▆▆▇▇▇▇██▇▇▇████████████████▇▇▇▇▇
val/loss,▆▇███▆▆▅▄▄▃▂▁▁▁▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂
best_val_acc,0.75159
best_val_acc_smoothed,0.75584
lr,0
train/acc,1
train/loss,0.56077


누적: [0.675, 0.873, 0.752]

80에폭   [0.713, 0.815, 0.694] 평균 0.741 폭 0.121
120에폭  [0.675, 0.873, 0.752] 평균 0.767 폭 0.198
차이 0.026


In [ ]:
import json, time
from src.ml.training.train import train

SPEAKERS = ["s01", "s03", "s04", "s05", "s06", "s07", "s08", "s09"]
SEEDS = [42, 1, 7]

LOG = DRIVE_ROOT / "cv60e120_results.json"
done = json.loads(LOG.read_text()) if LOG.exists() else []
seen = [tuple(x[:2]) for x in done]
print("이미 끝난 런", len(done), "/ 24")

t0 = time.time()
for sp in SPEAKERS:
    for sd in SEEDS:
        if (sp, sd) in seen:
            continue
        print("")
        print("========", sp, "· seed", sd, "· 60프레임 · 120에폭 ========")
        acc = train(
            manifest_path=manifest_f60, data_root=TRAIN_ROOT_F60,
            epochs=120, batch_size=16, learning_rate=2e-4, seed=sd,
            val_speakers=[sp],
            checkpoint_path=DRIVE_CHECKPOINTS / ("cv60e120_" + sp + "_seed" + str(sd) + ".pt"),
            num_workers=8, amp=True, ema_decay=0.998,
            hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
            wandb_project="lipreading", run_name="cv60e120_" + sp + "_seed" + str(sd))
        done.append([sp, sd, round(acc, 4)])
        LOG.write_text(json.dumps(done))
        print("저장 ·", len(done), "/ 24 · 경과", round((time.time() - t0) / 60), "분")
# end

이미 끝난 런 0 / 24

======== s01 · seed 42 · 60프레임 · 120에폭 ========


장치: cuda | 클래스: 15개
학습 1077개 · 검증 157개 클립 | 검증 화자 ['s01']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/120] train loss 2.7334 acc 0.076 | val loss 2.7102 acc 0.064 avg 0.064 | lr 2.00e-04
[  2/120] train loss 2.3964 acc 0.221 | val loss 2.7113 acc 0.064 avg 0.064 | lr 2.00e-04
[  3/120] train loss 2.0235 acc 0.375 | val loss 2.7168 acc 0.070 avg 0.066 | lr 2.00e-04
[  4/120] train loss 1.6729 acc 0.565 | val loss 2.7558 acc 0.146 avg 0.093 | lr 1.99e-04
[  5/120] train loss 1.3802 acc 0.736 | val loss 2.8431 acc 0.076 avg 0.098 | lr 1.99e-04
[  6/120] train loss 1.1217 acc 0.822 | val loss 2.9647 acc 0.076 avg 0.100 | lr 1.99e-04
[  7/120] train loss 0.9675 acc 0.873 | val loss 3.1168 acc 0.076 avg 0.076 | lr 1.98e-04
[  8/120] train loss 0.9148 acc 0.893 | val loss 3.2947 acc 0.076 avg 0.076 | lr 1.98e-04
[  9/120] train loss 0.8082 acc 0.928 |

lr,█████▇▇▇▇▇▆▆▆▆▆▅▅▅▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁
train/acc,▁▃▅▆▇███████████████████████████████████
train/loss,█▇▆▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▂▁▁▁▁▂▄▅▅▅▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇████████████
val/acc_smoothed,▁▁▁▁▁▁▂▃▃▄▄▅▆▇▇▇▇▇▇▇▇▇▇▇▇▇██████████████
val/loss,▅▆▇██▅▅▄▄▄▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_acc,0.76433
best_val_acc_smoothed,0.76645
lr,0
train/acc,1
train/loss,0.56051


저장 · 1 / 24 · 경과 19 분

======== s01 · seed 1 · 60프레임 · 120에폭 ========


장치: cuda | 클래스: 15개
학습 1077개 · 검증 157개 클립 | 검증 화자 ['s01']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/120] train loss 2.7189 acc 0.085 | val loss 2.7084 acc 0.070 avg 0.070 | lr 2.00e-04
[  2/120] train loss 2.4339 acc 0.182 | val loss 2.7149 acc 0.070 avg 0.070 | lr 2.00e-04
[  3/120] train loss 2.1741 acc 0.313 | val loss 2.7399 acc 0.070 avg 0.070 | lr 2.00e-04
[  4/120] train loss 1.8751 acc 0.454 | val loss 2.8010 acc 0.070 avg 0.070 | lr 1.99e-04
[  5/120] train loss 1.5238 acc 0.661 | val loss 2.9048 acc 0.070 avg 0.070 | lr 1.99e-04
[  6/120] train loss 1.2502 acc 0.756 | val loss 3.0463 acc 0.102 avg 0.081 | lr 1.99e-04
[  7/120] train loss 1.0555 acc 0.841 | val loss 3.2069 acc 0.121 avg 0.098 | lr 1.98e-04
[  8/120] train loss 0.9043 acc 0.912 | val loss 3.3363 acc 0.076 avg 0.100 | lr 1.98e-04
[  9/120] train loss 0.8251 acc 0.927 |

...

저장 · 16 / 24 · 경과 309 분

======== s07 · seed 1 · 60프레임 · 120에폭 ========


장치: cuda | 클래스: 15개
학습 1063개 · 검증 171개 클립 | 검증 화자 ['s07']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/120] train loss 2.7069 acc 0.107 | val loss 2.7077 acc 0.070 avg 0.070 | lr 2.00e-04
[  2/120] train loss 2.4224 acc 0.198 | val loss 2.7054 acc 0.082 avg 0.076 | lr 2.00e-04
[  3/120] train loss 2.0756 acc 0.368 | val loss 2.7209 acc 0.082 avg 0.078 | lr 2.00e-04
[  4/120] train loss 1.6935 acc 0.569 | val loss 2.7597 acc 0.088 avg 0.084 | lr 1.99e-04
[  5/120] train loss 1.3859 acc 0.720 | val loss 2.8254 acc 0.088 avg 0.086 | lr 1.99e-04
[  6/120] train loss 1.1708 acc 0.807 | val loss 2.9233 acc 0.088 avg 0.088 | lr 1.99e-04
[  7/120] train loss 0.9769 acc 0.875 | val loss 3.0492 acc 0.088 avg 0.088 | lr 1.98e-04
[  8/120] train loss 0.8620 acc 0.912 | val loss 3.1936 acc 0.058 avg 0.078 | lr 1.98e-04
[  9/120] train loss 0.7930 acc 0.944 |

lr,████████▇▇▇▇▇▇▇▆▆▆▆▆▅▅▅▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
train/acc,▁▆██████████████████████████████████████
train/loss,█▅▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▂▂▂▁▃▄▃▃▃▆▇▇███▇▆▆▆▇▇▇▇▇▆▆▅▅▅▅▅▅▅▅▄▄▄▄▄
val/acc_smoothed,▁▁▁▁▁▄▄▃▃▃▅▆███▆▆▇▇▇▇▇▇▇▇▆▅▅▅▅▅▅▅▅▅▄▄▄▄▄
val/loss,▅█▆▆▆▂▁▁▁▂▃▃▃▃▃▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▅
best_val_acc,0.38012
best_val_acc_smoothed,0.37817
lr,0
train/acc,1
train/loss,0.56046


저장 · 17 / 24 · 경과 328 분

======== s07 · seed 7 · 60프레임 · 120에폭 ========


장치: cuda | 클래스: 15개
학습 1063개 · 검증 171개 클립 | 검증 화자 ['s07']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/120] train loss 2.7455 acc 0.075 | val loss 2.7079 acc 0.064 avg 0.064 | lr 2.00e-04
[  2/120] train loss 2.4946 acc 0.166 | val loss 2.7124 acc 0.064 avg 0.064 | lr 2.00e-04
[  3/120] train loss 2.1358 acc 0.356 | val loss 2.7216 acc 0.064 avg 0.064 | lr 2.00e-04
[  4/120] train loss 1.7967 acc 0.534 | val loss 2.7491 acc 0.064 avg 0.064 | lr 1.99e-04
[  5/120] train loss 1.4408 acc 0.691 | val loss 2.8030 acc 0.094 avg 0.074 | lr 1.99e-04
[  6/120] train loss 1.1998 acc 0.794 | val loss 2.9012 acc 0.105 avg 0.088 | lr 1.99e-04
[  7/120] train loss 1.0240 acc 0.856 | val loss 3.0402 acc 0.070 avg 0.090 | lr 1.98e-04
[  8/120] train loss 0.8931 acc 0.899 | val loss 3.1946 acc 0.064 avg 0.080 | lr 1.98e-04
[  9/120] train loss 0.8301 acc 0.921 |

In [ ]:
import json, time
from src.ml.training.train import train

SPEAKERS = ["s01", "s03", "s04", "s05", "s06", "s07", "s08", "s09"]
SEEDS = [42, 1, 7]

LOG = DRIVE_ROOT / "cv60e120_results.json"
done = json.loads(LOG.read_text()) if LOG.exists() else []
seen = [tuple(x[:2]) for x in done]
print("이미 끝난 런", len(done), "/ 24")

t0 = time.time()
for sp in SPEAKERS:
    for sd in SEEDS:
        if (sp, sd) in seen:
            continue
        print("")
        print("========", sp, "· seed", sd, "· 60프레임 · 120에폭 ========")
        acc = train(
            manifest_path=manifest_f60, data_root=TRAIN_ROOT_F60,
            epochs=120, batch_size=16, learning_rate=2e-4, seed=sd,
            val_speakers=[sp],
            checkpoint_path=DRIVE_CHECKPOINTS / ("cv60e120_" + sp + "_seed" + str(sd) + ".pt"),
            num_workers=8, amp=True, ema_decay=0.998,
            hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
            wandb_project="lipreading", run_name="cv60e120_" + sp + "_seed" + str(sd))
        done.append([sp, sd, round(acc, 4)])
        LOG.write_text(json.dumps(done))
        print("저장 ·", len(done), "/ 24 · 경과", round((time.time() - t0) / 60), "분")
# end

In [ ]:
import unicodedata
import numpy as np
import torch
from pathlib import Path
from torch import nn

from src.ml.models.lip_reading_model import LipReadingModel
from src.ml.models.backbone import LipReadingBackbone
from src.ml.models.temporal import TemporalBiGRU
from src.ml.models.classification_head import ClassificationHead
from src.ml.training.dataset import LipReadingDataset
from src.ml.training import train as T

AUX_DIM = 44
PROJ_DIM = 128

_z = np.load(DRIVE_ROOT / "aux_f60.npz", allow_pickle=False)
AUX_FEATS = _z["feats"].astype(np.float32)
AUX_INDEX = dict(zip(list(_z["names"]), range(len(_z["names"]))))
print("보조 특징", AUX_FEATS.shape)

if not hasattr(LipReadingDataset, "_orig_getitem"):
    LipReadingDataset._orig_getitem = LipReadingDataset.__getitem__
if not hasattr(LipReadingModel, "_orig_init"):
    LipReadingModel._orig_init = LipReadingModel.__init__
    LipReadingModel._orig_forward = LipReadingModel.forward
if not hasattr(T, "_orig_run_epoch"):
    T._orig_run_epoch = T.run_epoch

MISS = []

def getitem_aux(self, index):
    frames, label = LipReadingDataset._orig_getitem(self, index)
    name = unicodedata.normalize("NFC", Path(self.rows[index]["clip_path"]).stem)
    j = AUX_INDEX.get(name, -1)
    if j < 0:
        MISS.append(name)
        a = np.zeros((frames.shape[1], AUX_DIM), dtype=np.float32)
    else:
        a = AUX_FEATS[j]
    return frames, torch.from_numpy(a), label

def init_aux(self, num_classes, hidden_dim=256, num_layer=2, dropout=0.2,
             pretrained=False, freeze_backbone=False):
    nn.Module.__init__(self)
    self.backbone = LipReadingBackbone(pretrained=pretrained)
    if freeze_backbone:
        self.backbone.freeze_resnet()
    self.aux_proj = nn.Sequential(nn.Linear(AUX_DIM, PROJ_DIM), nn.ReLU())
    self.temporal = TemporalBiGRU(
        input_dim=self.backbone.feature_dim + PROJ_DIM,
        hidden_dim=hidden_dim, num_layer=num_layer, dropout=dropout)
    self.head = ClassificationHead(
        input_dim=self.temporal.output_dim, num_classes=num_classes, dropout=dropout)

def forward_aux(self, frames, aux):
    f = self.backbone(frames)
    f = torch.cat([f, self.aux_proj(aux)], dim=2)
    return self.head(self.temporal(f))

def run_epoch_aux(model, loader, criterion, device, optimizer=None,
                  amp=False, grad_clip=None, averager=None):
    is_training = optimizer is not None
    model.train(is_training)
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    with torch.set_grad_enabled(is_training):
        for frames, aux, labels in loader:
            frames = frames.to(device, non_blocking=True)
            aux = aux.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=amp):
                logits = model(frames, aux)
                loss = criterion(logits, labels)
            if is_training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                if grad_clip:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
                if averager is not None:
                    averager.update(model)
            total_loss += loss.float().item() * labels.size(0)
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_count += labels.size(0)
    return total_loss / total_count, total_correct / total_count

LipReadingDataset.__getitem__ = getitem_aux
LipReadingModel.__init__ = init_aux
LipReadingModel.forward = forward_aux
T.run_epoch = run_epoch_aux
print("패치 적용 완료")
# end

In [ ]:
res = []
for sp in ["s05", "s06"]:
    print("")
    print("======== 랜드마크 추가 ·", sp, "· seed 42 ========")
    acc = T.train(
        manifest_path=manifest_f60, data_root=TRAIN_ROOT_F60,
        epochs=80, batch_size=16, learning_rate=2e-4, seed=42,
        val_speakers=[sp],
        checkpoint_path=DRIVE_CHECKPOINTS / ("aux_" + sp + ".pt"),
        num_workers=8, amp=True, ema_decay=0.998,
        hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
        wandb_project="lipreading", run_name="aux_" + sp)
    res.append([sp, round(acc, 4)])
    print("누적:", res)
print("")
print("좌표 못 찾은 클립:", len(set(MISS)))
# end